In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Preprocessing

In [ ]:
from preprocessing import read_flow
bom_dir = 'BOM_CD3_01DEC25'
bom_flow, bom_samples, bom_session = read_flow(bom_dir, "BOM")

In [ ]:
lln_dir = 'LLN_CD3_01DEC25'
lln_flow, lln_samples, lln_session = read_flow(lln_dir, 'LLN')

In [ ]:
lng_dir = 'LNG_CD3_01DEC25'
lng_flow, lng_samples, lng_session = read_flow(lng_dir, 'LNG')

In [ ]:
mln_dir = 'MLN_CD3_01DEC25'
mln_flow, mln_samples, mln_session = read_flow(mln_dir, 'MLN')

In [ ]:
spl_dir = 'SPL_CD3_01DEC25'
spl_flow, spl_samples, spl_session = read_flow(spl_dir, 'SPL')

In [ ]:
df_flow = pd.concat([bom_flow, lln_flow, lng_flow, mln_flow, spl_flow])

In [ ]:
exclude = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-B-A', 'SSC-B-H',
           'SSC-H', 'AF-A', 'CD66bCD19CD326LD', 'Time', 'CD45', 'Event #']
df_flow = df_flow.drop(columns=exclude)

In [ ]:
df_flow_counts = df_flow[
    df_flow.select_dtypes(include=[np.number]).columns.difference(["age"])
]

In [ ]:
from preprocessing import pd_to_adata
adata = pd_to_adata(df_flow, df_flow_counts)

In [ ]:
adata.X = np.arcsinh(adata.X / 150)

In [ ]:
sc.pp.scale(adata, max_value=3)

In [ ]:
adata = adata[(adata[:, 'CD3'].X > 0)]
adata = adata[:, adata.var.index != 'CD3']

In [ ]:
adata.obs.rename(columns={'TCRVaJa': 'TCRva'}, inplace=True)

In [ ]:
from preprocessing import population_filter
CD4 = population_filter(adata, 'CD4', 0.0)

In [ ]:
CD8 = population_filter(adata, 'CD8', 0.0)

In [ ]:
from run_pipeline import clustering_pipeline
CD4 = clustering_pipeline(CD4, sample_name='CD4', tissue_type='All Tissues')

## IL33R Expression

In [ ]:
sc.pl.violin(CD4, 'IL33R', groupby='group',
             stripplot=False, order=['ctr', 'hst', 'ftl'])

In [ ]:
sc.pl.violin(CD4, 'IL33R', groupby='asthma',
             stripplot=False, order=['control', 'asthmatic'])

In [ ]:
marker_genes = {
    'Tfh': ['CXCR5', 'PD-1'],
    'Th1': ['CXCR3'],
    'Th2': ['CRTH2'],
    'Trm': ['CD103', 'CD69'],
    'Treg': ['CD25', 'FOXP3'],
    'Memory': ['CCR7', 'CD45RA']
}

In [ ]:
from run_pipeline import dem_ranked
CD4, unique_values = dem_ranked(CD4)

In [ ]:
# celltype = {'celltype': []}
# cluster_to_genes = {
#     '15': 'Effector Memory (15)',
#     '16': 'Effector Memory CD103+CD69+ (16)'
#     ''
# }
# celltype['celltype'] = [cluster_to_genes[leiden]
#                         for leiden in sample.obs['leiden']]
# sample.obs["celltype"] = celltype['celltype']

# print(sample.obs[["leiden", "celltype"]].head())

In [ ]:
# sc.tl.rank_genes_groups(sample, 'leiden', groups=[
#                         '16'], reference='15', method='wilcoxon')
# sc.pl.rank_genes_groups(sample, groups=['16'], n_genes=20)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from plotting_methods import annotated_umap

annotated_umap(CD4, "All Tissues", "CD4", obs='celltype')

In [ ]:
for group in ['ctr', 'hst', 'ftl']:
    adata_group = CD4[CD4.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['leiden'],
        title=f'{"All Tissues"} {"CD4"} {group} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in CD4.obs['tissue'].unique().tolist():
    adata_group = CD4[CD4.obs['tissue'] == tissue]

    sc.pl.umap(
        adata_group,
        color=['leiden'],
        title=f'{"All Tissues"} {"CD4"} {tissue} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in CD4.obs['tissue'].unique():
    for group in ['ctr', 'hst', 'ftl']:

        mask = (
            (CD4.obs['tissue'] == tissue) &
            (CD4.obs['group'] == group)
        )
        adata_group = CD4[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='celltype',
            title=f'{"CD4"} {tissue} {group} celltypes',
            cmap='turbo',
            show=False
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
for group in ['ctr', 'hst', 'ftl']:
    adata_group = CD4[CD4.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['age'],
        title=f'{"All Tissues"} {"CD4"} {group} age',
        cmap='turbo',
        show=False,
        vmin=0,
        vmax=70
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in CD4.obs['tissue'].unique():
    for group in ['ctr', 'hst', 'ftl']:

        mask = (
            (CD4.obs['tissue'] == tissue) &
            (CD4.obs['group'] == group)
        )
        adata_group = CD4[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='age',
            title=f'{"CD4"} {tissue} {group} age',
            cmap='turbo',
            show=False,
            vmin=0,
            vmax=70
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
for group in ['control', 'asthmatic',]:
    adata_group = CD4[CD4.obs['asthma'] == group]

    sc.pl.umap(
        adata_group,
        color=['celltype'],
        title=f'{"All Tissues"} {"CD4"} {group} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in CD4.obs['tissue'].unique():
    for group in ['control', 'asthmatic']:

        mask = (
            (CD4.obs['tissue'] == tissue) &
            (CD4.obs['asthma'] == group)
        )
        adata_group = CD4[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='celltype',
            title=f'{"CD4"} {tissue} {group} celltypes',
            cmap='turbo',
            show=False
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
for group in ['control', 'asthmatic',]:
    adata_group = CD4[CD4.obs['asthma'] == group]

    sc.pl.umap(
        adata_group,
        color=['age'],
        title=f'{"All Tissues"} {"CD4"} {group} age',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in CD4.obs['tissue'].unique():
    for group in ['control', 'asthmatic']:

        mask = (
            (CD4.obs['tissue'] == tissue) &
            (CD4.obs['asthma'] == group)
        )
        adata_group = CD4[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='age',
            title=f'{"CD4"} {tissue} {group} age',
            cmap='turbo',
            show=False,
            vmin=0,
            vmax=70,
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
plt.rcParams.update({'font.size': 12})

In [ ]:
sc.pl.dotplot(CD4, list(adata.var_names), swap_axes=True, groupby='leiden', title="{} {} Dotplot".format(
    "All Tissues", "CD4"), cmap='RdBu_r', vmin=-5, vmax=5, dendrogram=True)

In [ ]:
from plotting_methods import composition_dotplot
composition_dotplot(CD4, group_x='tissue', group_y='celltype')

In [ ]:
composition_dotplot(CD4, group_x='group', group_y='celltype')

In [ ]:
composition_dotplot(CD4, group_x='asthma', group_y='celltype')

In [ ]:
sc.pl.stacked_violin(CD4, list(adata.var_names), swap_axes=True, groupby='leiden', title="{} {} Violinplot".format(
    "All Tissues", "CD4"), cmap='RdBu_r', vmin=-5, vmax=5)